In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_IMPAIRMENT_ONLY.csv'),
                 encoding='latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)

target = 'INJ_SEV'

categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION', 'VSPD_LIM', 'VE_TOTAL', 'PEDS' # , 'AIR_BAG','EJECTION', 'ATST_TYP',
] 
numeric_cols = ['TRAV_SP', 'AGE']

for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

params = {
    'objective': 'multiclass',
    'num_class': len(y.unique()),
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 100,
    'random_state': 42,
    'verbose': -1
}


train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=train_data)


model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, valid_data],
    num_boost_round=1000,

)


y_pred = model.predict(X_test)
y_pred_class = np.argmax(y_pred, axis=1)

accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {accuracy:.2f}\n")


balanced_accuracy = balanced_accuracy_score(y_test, y_pred_class)
print(f"Balanced Accuracy: {balanced_accuracy:.2f}\n")

print("Confusion Matrix:")
print

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_class)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_test, y_pred_class))

In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model)

shap_sample = X_test.iloc[:100] 

shap_values = explainer.shap_values(shap_sample)

shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)
